In [2]:
import subprocess
import os
import geopandas as gpd
from pyproj import CRS
from tqdm import tqdm
import time
import fiona
import pandas as pd

# Connect to the Database

In [3]:
# Database connection (no username/password required)
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "cambridge_growth_company"
pg_conn_string = f"PG:host={db_host} dbname={db_name} port={db_port}"

# User Input Required

In [4]:
# Replace with the actual path to your CSV file
csv_file = r"N:\2207_Cambridge Growth Company\WORKING\Excel\2207_Cambridge_Baseline Evidence Base_Data list_v1.csv"

# Read the CSV into a DataFrame
df = pd.read_csv(csv_file)

# Create dictionary from 'Upload name' and 'File Path' columns
upload_dict = dict(zip(df['Upload name'], df['File Path']))

print(upload_dict)

{'pla_gcsp_district_local_or_neighbourhood_centre_april_2025': 'N:\\2207_Cambridge Growth Company\\Source IN\\250430_GCSP_Data upload 01\\City\\District, Local or Neighbourhood Centre\\District__Local_or_Neighbourhood_Centre.shp', 'pla_gcsp_city_centre_april_2025': 'N:\\2207_Cambridge Growth Company\\Source IN\\250430_GCSP_Data upload 01\\City\\City Centre\\City_Centre.shp', 'pla_gcsp_shopping_frontage_april_2025': 'N:\\2207_Cambridge Growth Company\\Source IN\\250430_GCSP_Data upload 01\\City\\Shopping Frontage\\Shopping_Frontage.shp', 'pla_gcsp_primary_shopping_area_april_2025': 'N:\\2207_Cambridge Growth Company\\Source IN\\250430_GCSP_Data upload 01\\City\\Primary Shopping Area\\Primary_Shopping_Area.shp', 'pla_gcsp_protected_village_amenity_area_april_2025': 'N:\\2207_Cambridge Growth Company\\Source IN\\250430_GCSP_Data upload 01\\SCDC\\Protected Village Amenity Area\\Protected_Village_Amenity_Area.shp', 'pla_gcsp_listed_buildings_april_2025': 'N:\\2207_Cambridge Growth Company\\

In [5]:
#Input target projection (CRS). #Change to None if you dont know input projection
target_crs = CRS.from_epsg(27700)

#Define path the ogr2ogr.exe
ogr2ogr_path = r"C:\Program Files\QGIS 3.34.11\bin\ogr2ogr.exe"

# DO NOT CHANGE
gpkg_path = r"C:\Users\abhimanya.achara\OneDrive - priorpartners.com\Documents\gdb_tem_layer\temp_upload.gpkg"  # temp file for faster upload

In [6]:
# ---- Loop through each layer ---- #
for layer_name, input_path in upload_dict.items():
    print(f"\n=== Processing: {layer_name} ===")

    # STEP 1: Load & Inspect CRS
    print("Reading input layer...")
    #gdf = gpd.read_file(input_path, engine="pyogrio") #uncomment for large files
    gdf = gpd.read_file(input_path)

    if not gdf.crs:
        raise ValueError(f"{input_path} does not have a valid CRS defined.")

    input_epsg = gdf.crs.to_epsg()
    target_epsg = target_crs.to_epsg()

    print(f"Detected input CRS: {input_epsg}")
    print(f"Target CRS: {target_epsg}")

    # STEP 2: Reproject and/or Validate Geometries Only If Needed
    needs_reproject = input_epsg != target_epsg
    needs_validation = not gdf.is_valid.all()

    if needs_reproject:
        print("Reprojecting to target CRS...")
        gdf = gdf.to_crs(target_crs)

    if needs_validation:
        print("Fixing invalid geometries using buffer(0)...")
        gdf["geometry"] = gdf.buffer(0)

    print(f"Writing to GeoPackage at {gpkg_path} (layer: {layer_name})...")
    gdf.to_file(gpkg_path, driver="GPKG", layer=layer_name)

    # STEP 3: Upload to PostGIS
    print("Uploading to PostGIS via ogr2ogr...")
    start_time = time.time()

    ogr2ogr_cmd = [
        ogr2ogr_path,
        "-f", "PostgreSQL",
        pg_conn_string,
        gpkg_path,
        "-nln", f"{db_schema}.{layer_name}",
        "-nlt", "PROMOTE_TO_MULTI",
        "-lco", "GEOMETRY_NAME=geometry",
        "-lco", "SCHEMA=" + db_schema,
        "-lco", "OVERWRITE=YES",
        "-overwrite",
        "-gt", "10000",  # Fixed comma
        "-progress"
    ]

    with tqdm(total=len(gdf), desc=f"Uploading {layer_name}", unit="features", bar_format="{l_bar}{bar} | {n_fmt}/{total_fmt} features | {elapsed}") as pbar:
        result = subprocess.run(ogr2ogr_cmd, capture_output=True, text=True)
        pbar.update(len(gdf))

    if result.returncode != 0:
        print(f"\nError uploading {layer_name}:")
        print(result.stderr)
    else:
        print("Upload completed successfully.")
        print(f"Time taken: {time.time() - start_time:.2f} seconds")


=== Processing: pla_gcsp_district_local_or_neighbourhood_centre_april_2025 ===
Reading input layer...
Detected input CRS: 27700
Target CRS: 27700
Writing to GeoPackage at C:\Users\abhimanya.achara\OneDrive - priorpartners.com\Documents\gdb_tem_layer\temp_upload.gpkg (layer: pla_gcsp_district_local_or_neighbourhood_centre_april_2025)...


C:\Users\abhimanya.achara\Anaconda3\lib\site-packages\geopandas\io\file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


Uploading to PostGIS via ogr2ogr...


Uploading pla_gcsp_district_local_or_neighbourhood_centre_april_2025: 100%|██████████ | 28/28 features | 00:24
C:\Users\abhimanya.achara\Anaconda3\lib\site-packages\geopandas\io\file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


Upload completed successfully.
Time taken: 24.94 seconds

=== Processing: pla_gcsp_city_centre_april_2025 ===
Reading input layer...
Detected input CRS: 27700
Target CRS: 27700
Writing to GeoPackage at C:\Users\abhimanya.achara\OneDrive - priorpartners.com\Documents\gdb_tem_layer\temp_upload.gpkg (layer: pla_gcsp_city_centre_april_2025)...
Uploading to PostGIS via ogr2ogr...


...

Upload completed successfully.
Time taken: 35.86 seconds

=== Processing: prop_gcsp_helaa_published_sites_nov_2021 ===
Reading input layer...
Detected input CRS: 27700
Target CRS: 27700
Fixing invalid geometries using buffer(0)...
Writing to GeoPackage at C:\Users\abhimanya.achara\OneDrive - priorpartners.com\Documents\gdb_tem_layer\temp_upload.gpkg (layer: prop_gcsp_helaa_published_sites_nov_2021)...


C:\Users\abhimanya.achara\Anaconda3\lib\site-packages\geopandas\io\file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


Uploading to PostGIS via ogr2ogr...


Uploading prop_gcsp_helaa_published_sites_nov_2021: 100%|██████████ | 728/728 features | 00:46

Upload completed successfully.
Time taken: 46.37 seconds
